In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
CREATE WIDGET TEXT CATALOG default "adbdep_0013";
CREATE WIDGET TEXT STORAGE default "adlsproject13dev2";

In [0]:
-- usuario
SELECT current_user();

In [0]:
SELECT DISTINCT
    origin.pipeline_name,
    origin.pipeline_id,
    origin.update_id,
    origin.flow_name
FROM event_log(TABLE(control.processed_files))
WHERE event_type = 'flow_progress'
ORDER BY origin.update_id DESC;

In [0]:
CREATE TABLE IF NOT EXISTS IDENTIFIER(:CATALOG || '.control.etl_run_log') (
    run_id STRING,
    pipeline_name STRING,
    flow_name STRING,
    layer STRING,
    object_name string,
    start_ts TIMESTAMP,
    end_ts TIMESTAMP,
    duration_sec BIGINT,
    status STRING,
    rows_written BIGINT,
    created_at TIMESTAMP,
    error_message STRING,
    error_detail STRING
)
USING DELTA;

In [0]:
MERGE INTO IDENTIFIER(:CATALOG || '.control.etl_run_log') AS tgt

USING (

    WITH flow_events AS (

        SELECT
            origin.update_id AS run_id,
            origin.pipeline_name AS pipeline_name,
            origin.flow_name AS flow_name,

            timestamp,

            details:flow_progress.status::STRING AS status,

            TRY_CAST(
                details:flow_progress.metrics.num_output_rows
                AS BIGINT
            ) AS rows_written

        FROM event_log(TABLE(control.processed_files))

        WHERE event_type = 'flow_progress'
          AND origin.flow_name IS NOT NULL
          AND origin.flow_name NOT LIKE 'pipelines.%'
    ),

    flow_metrics AS (

        SELECT
            run_id,
            pipeline_name,
            flow_name,

            MIN(timestamp) AS start_ts,
            MAX(timestamp) AS end_ts,
            MAX(rows_written) AS rows_written

        FROM flow_events

        GROUP BY
            run_id,
            pipeline_name,
            flow_name
    ),

    flow_status AS (

        SELECT
            run_id,
            flow_name,
            status

        FROM (
            SELECT
                run_id,
                flow_name,
                status,

                ROW_NUMBER() OVER (
                    PARTITION BY run_id, flow_name
                    ORDER BY timestamp DESC
                ) AS rn

            FROM flow_events

            WHERE status IS NOT NULL
        )

        WHERE rn = 1
    ),
        error_events AS (

        SELECT
            origin.update_id AS run_id,
            origin.flow_name AS flow_name,

            timestamp AS error_ts,

            message AS error_message,

            error AS error_detail,

            ROW_NUMBER() OVER (
                PARTITION BY
                    origin.update_id,
                    origin.flow_name
                ORDER BY timestamp DESC
            ) AS rn

        FROM event_log(TABLE(control.processed_files))

        WHERE
            level = 'ERROR'
            OR error IS NOT NULL
    ),
        latest_error AS (
            SELECT
                run_id,
                flow_name,
                error_message,
                error_detail

            FROM error_events

            WHERE rn = 1
    )

    SELECT
        m.run_id,
        m.pipeline_name,
        m.flow_name,

        CASE
            WHEN m.flow_name LIKE '%.ly_bronze.%' THEN 'BRONZE'
            WHEN m.flow_name LIKE '%.ly_silver.%' THEN 'SILVER'
            WHEN m.flow_name LIKE '%.ly_gold.%' THEN 'GOLD'
            WHEN m.flow_name LIKE '%.control.%' THEN 'CONTROL'
            ELSE 'OTHER'
        END AS layer,

        element_at(
            split(m.flow_name, '\\.'),
            -1
        ) AS object_name,

        m.start_ts,
        m.end_ts,

        TIMESTAMPDIFF(
            SECOND,
            m.start_ts,
            m.end_ts
        ) AS duration_sec,

        s.status,
        m.rows_written,

        current_timestamp() AS created_at,
        e.error_message,
        e.error_detail

    FROM flow_metrics m

    LEFT JOIN flow_status s
        ON m.run_id = s.run_id
        AND m.flow_name = s.flow_name

    LEFT JOIN latest_error e
    ON m.run_id = e.run_id
    AND m.flow_name = e.flow_name

    WHERE m.pipeline_name = 'ingest_y_transform'

) AS src

ON  tgt.run_id = src.run_id
AND tgt.flow_name = src.flow_name

WHEN MATCHED THEN UPDATE SET
    tgt.pipeline_name = src.pipeline_name,
    tgt.layer = src.layer,
    tgt.object_name = src.object_name,
    tgt.start_ts = src.start_ts,
    tgt.end_ts = src.end_ts,
    tgt.duration_sec = src.duration_sec,
    tgt.status = src.status,
    tgt.rows_written = src.rows_written,
    tgt.error_message = src.error_message,
    tgt.error_detail = src.error_detail
WHEN NOT MATCHED THEN INSERT (
    run_id,
    pipeline_name,
    flow_name,
    layer,
    object_name,
    start_ts,
    end_ts,
    duration_sec,
    status,
    rows_written,
    created_at,
    error_message,
    error_detail
)
VALUES (
    src.run_id,
    src.pipeline_name,
    src.flow_name,
    src.layer,
    src.object_name,
    src.start_ts,
    src.end_ts,
    src.duration_sec,
    src.status,
    src.rows_written,
    src.created_at,
    src.error_message,
    src.error_detail
);